# RDF — DOPC/DOPA composition series

2D lateral radial distribution function g(r) between lipid **phosphate (P)** atoms,
for the DOPC/DOPA bilayer at five mixtures:

| DOPC/DOPA | DOPA % |
|-----------|--------|
| 65/35 | 35 |
| 72/28 | 28 |
| 80/20 | 20 |
| 87/13 | 13 |
| 95/5  | 5  |

Three pair panels: **DOPC-DOPC, DOPA-DOPA, DOPC-DOPA**, each overlaying the five
compositions (colour gradient) with a shaded **error band from block averaging**.

Two correctness choices built in:
* g(r) is computed **per leaflet** (upper/lower separately, then combined) - a plain
  3D or naive xy RDF would mix cross-leaflet pairs and distort the first peak.
* the band is the **std across N contiguous time blocks** (set `NBLOCKS`). With a
  single frame there is no band.


## Setup

In [ ]:
# !pip install "MDAnalysis>=2.4" numpy matplotlib
%matplotlib inline
import os, numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

## Config - point each composition at its files

One simulation per mixture, so give the topology + trajectory for each. Use the
reduced trajectories from `shrink_trajectory.sh` if the full `.xtc` is too big
(headgroup-only `heads_skip.xtc` + `heads.tpr` is ideal for RDF).

In [ ]:
HEAD    = "P"
RMAX_NM = 4.0          # x-axis range (nm), matches the reference figure
BIN_NM  = 0.02         # bin width (nm)
NBLOCKS = 5            # time blocks for the error band
STEP    = 1            # frame stride

# label -> (topology, trajectory, DOPA_percent).  EDIT THE PATHS.
SYSTEMS = {
    "65/35": ("DOPC_DOPA_65_35/md.tpr", "DOPC_DOPA_65_35/heads_skip.xtc", 35),
    "72/28": ("DOPC_DOPA_72_28/md.tpr", "DOPC_DOPA_72_28/heads_skip.xtc", 28),
    "80/20": ("DOPC_DOPA_80_20/md.tpr", "DOPC_DOPA_80_20/heads_skip.xtc", 20),
    "87/13": ("DOPC_DOPA_87_13/md.tpr", "DOPC_DOPA_87_13/heads_skip.xtc", 13),
    "95/5" : ("DOPC_DOPA_95_5/md.tpr",  "DOPC_DOPA_95_5/heads_skip.xtc",  5),
}

PAIRS = [("DOPC","DOPC"), ("DOPA","DOPA"), ("DOPC","DOPA")]
YLIM  = {("DOPC","DOPC"):(0.5,1.5), ("DOPA","DOPA"):(0.5,1.5), ("DOPC","DOPA"):(0.75,1.25)}
os.makedirs("results", exist_ok=True)

## RDF core (per-leaflet, block-averaged)

In [ ]:
def _pdist2d(a_xy, b_xy, Lx, Ly, same):
    """Minimum-image lateral distances (A) between two point sets."""
    if len(a_xy)==0 or len(b_xy)==0:
        return np.empty(0)
    dx = a_xy[:,None,0]-b_xy[None,:,0]; dy = a_xy[:,None,1]-b_xy[None,:,1]
    dx -= Lx*np.round(dx/Lx); dy -= Ly*np.round(dy/Ly)
    r = np.sqrt(dx*dx+dy*dy)
    return r[np.triu_indices(r.shape[0],1)] if same else r.ravel()

def rdf2d_blocks(u, resA, resB, head=HEAD, rmax_nm=RMAX_NM, bin_nm=BIN_NM,
                 nblocks=NBLOCKS, step=STEP):
    same = (resA==resB)
    rmax = rmax_nm*10.0; nbins = int(round(rmax/(bin_nm*10.0)))
    edges = np.linspace(0, rmax, nbins+1); ctr = 0.5*(edges[1:]+edges[:-1])
    shell = np.pi*(edges[1:]**2 - edges[:-1]**2)
    selA = u.select_atoms(f"name {head} and resname {resA}")
    selB = u.select_atoms(f"name {head} and resname {resB}")
    allP = u.select_atoms(f"name {head}")
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    gblocks = []
    for blk in blocks:
        hist = np.zeros(nbins); norm = 0.0
        for fi in blk:
            ts = u.trajectory[int(fi)]
            Lx, Ly = ts.dimensions[0], ts.dimensions[1]; area = Lx*Ly
            zmid = allP.positions[:,2].mean()
            Apos, Bpos = selA.positions, selB.positions
            for sign in (True, False):                      # upper then lower leaflet
                a = Apos[(Apos[:,2]>=zmid)==sign][:,:2]
                b = Bpos[(Bpos[:,2]>=zmid)==sign][:,:2]
                hist += np.histogram(_pdist2d(a,b,Lx,Ly,same), bins=edges)[0]
                Na, Nb = len(a), len(b)
                norm += 0.5*Na*((Nb-1)/area) if same else Na*(Nb/area)
        if norm > 0:
            gblocks.append(hist/(norm*shell))
    G = np.array(gblocks)
    mean = G.mean(axis=0)
    std  = G.std(axis=0) if len(G) > 1 else np.zeros_like(mean)
    return ctr/10.0, mean, std      # r in nm

## Compute for every composition and pair

In [ ]:
results = {(a,b): {} for a,b in PAIRS}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})"); continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for a, b in PAIRS:
        r, m, s = rdf2d_blocks(u, a, b)
        results[(a,b)][label] = (r, m, s)
        np.savetxt(f"results/rdf_{a}-{b}_{label.replace('/','-')}.dat",
                   np.column_stack([r,m,s]), header="r_nm  g(r)  std")
print("done")

## Plot - reference-figure style

Colour gradient runs light->dark with **decreasing DOPA %** (darkest = 5%). Shaded
band = +/-1 std across blocks. Italic pair label sits inside each panel.

In [ ]:
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])   # high DOPA -> low
cmap = plt.get_cmap("Reds")
n = max(2, len(labels_sorted))
colors = {L: cmap(0.25 + 0.7*i/(n-1)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, len(PAIRS), figsize=(4.2*len(PAIRS), 3.6))
if len(PAIRS)==1: axes=[axes]
for ax, (a,b) in zip(axes, PAIRS):
    for L in labels_sorted:
        if L not in results[(a,b)]: continue
        r, m, s = results[(a,b)][L]
        ax.fill_between(r, m-s, m+s, color=colors[L], alpha=0.25, lw=0)
        ax.plot(r, m, color=colors[L], lw=1.6,
                label=f"{L}  ({SYSTEMS[L][2]}% DOPA)")
    ax.axhline(1.0, color="0.6", lw=0.6, ls="--")
    ax.set_xlim(0, RMAX_NM); ax.set_ylim(*YLIM.get((a,b),(0.5,1.5)))
    ax.set_xlabel("Distance (nm)")
    ax.text(0.95, 0.06, f"{a}-{b}", transform=ax.transAxes, style="italic",
            ha="right", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)
axes[0].set_ylabel("RDF")
handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=n,
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.12))
fig.suptitle("DOPC/DOPA lateral RDF (P-P) vs composition", y=1.18)
plt.tight_layout()
plt.savefig("results/rdf_DOPC_DOPA_composition.png", dpi=200, bbox_inches="tight")
plt.show()

## GROMACS route (equivalent, per composition)

`gmx rdf` with `-xy` gives the in-plane RDF. Run it once per composition and per
pair. Note `gmx rdf -xy` does **not** split leaflets, so its first peak differs
slightly from the per-leaflet Python result above.

```bash
# one composition (repeat for 65_35, 72_28, 80_20, 87_13, 95_5):
D=DOPC_DOPA_65_35

# DOPC-DOPC
gmx rdf -s $D/md.tpr -f $D/md.xtc -n $D/index.ndx \
    -ref "name P and resname DOPC" -sel "name P and resname DOPC" \
    -xy -bin 0.02 -rmax 4 -o $D/rdf_DOPC-DOPC.xvg

# DOPA-DOPA
gmx rdf -s $D/md.tpr -f $D/md.xtc -n $D/index.ndx \
    -ref "name P and resname DOPA" -sel "name P and resname DOPA" \
    -xy -bin 0.02 -rmax 4 -o $D/rdf_DOPA-DOPA.xvg

# DOPC-DOPA
gmx rdf -s $D/md.tpr -f $D/md.xtc -n $D/index.ndx \
    -ref "name P and resname DOPC" -sel "name P and resname DOPA" \
    -xy -bin 0.02 -rmax 4 -o $D/rdf_DOPC-DOPA.xvg
```

**Error band from GROMACS:** run each `gmx rdf` on time blocks with `-b/-e`
(e.g. 5 windows) and take the mean/std across the `.xvg` files, or just use the
Python block averaging above.


## (Optional) plot straight from gmx .xvg files

In [ ]:
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#","@"))]
    d = np.array(rows); return d[:,0], d[:,1]     # r(nm), g(r)

# Example: overlay one pair from gmx outputs (fill in your paths):
# gmx_files = {"65/35":"DOPC_DOPA_65_35/rdf_DOPC-DOPA.xvg"}
# plt.figure(figsize=(4.5,3.6))
# for L in labels_sorted:
#     if L in gmx_files:
#         r,g = read_xvg_xy(gmx_files[L]); plt.plot(r,g,color=colors[L],label=L)
# plt.axhline(1,ls="--",color="0.6",lw=.6); plt.xlim(0,4); plt.ylim(0.75,1.25)
# plt.xlabel("Distance (nm)"); plt.ylabel("RDF"); plt.legend(); plt.show()

## Notes

* **Reduced trajectories:** for RDF you only need P atoms, so `heads_skip.xtc` +
  `heads.tpr` from `shrink_trajectory.sh` are perfect and tiny.
* **Bin/rmax** are set to 0.02 nm and 4 nm to match the figure; raise `NBLOCKS`
  for a smoother band if you have many frames.
* **Convergence:** the DOPA-DOPA curve is the noisiest (fewest atoms, especially
  at 5%); use more frames / a longer trajectory there.
* **Single frame:** with only a `.gro` you get one curve and no band; a trajectory
  is needed for meaningful RDF statistics.
